In [106]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import pymupdf
import requests
import os
from datetime import datetime

In [131]:
dir_path = Path("data/landing/")
latest_file = max(dir_path.glob("*.parquet"), key=lambda f: f.stat().st_mtime)

print(latest_file.name)

16-04-2026.parquet


In [ ]:
table = pq.read_table(latest_file)
df = table.to_pandas()

In [134]:
df.rename(columns={
    'Код\nИнструмента' : 'Код инструмента',
    'Наименование\nИнструмента' : 'Наименование Инструмента',
    'Базис\nпоставки' : 'Базис поставки',
    'Объем\nДоговоров\nв единицах\nизмерения' : 'Объем договоров в ед',
    'Обьем\nДоговоров,\nруб.' : 'Объем договоров руб',
    'Изменение рыночной\nцены к цене\nпредыдуего дня' : 'Изменение рыночной цены к цене предыдуего дня руб)',
    'Col6' : 'Изменение рыночной цены к цене предыдуего дня %)', 
    'Цена (за единицу измерения), руб.' : 'Цена мин',
    'Col8' : 'Цена сред',
    'Col9' : 'Цена макс',
    'Col10' : 'Цена рын',
    'Цена в Заявках (за единицу\nизмерения)' : 'Цена в заявках (лучшее предложение)',
    'Col12' : 'Цена в заявках (Лучший спрос)',
    'Количество\nДоговоров,\nшт.' : 'Количество договоров'
}, inplace=True)

In [135]:
dfc = df.copy()

In [136]:
mask = (dfc["Количество договоров"].isna()) | \
(dfc["Количество договоров"] == "-") | \
(dfc["Количество договоров"] == "0") 

In [137]:
dfc = dfc[~mask].reset_index(drop=True)

In [138]:
col = 'Изменение рыночной цены к цене предыдуего дня руб)'
dfc[col] = dfc[col].astype(str).replace('-', '0').astype(float)

In [139]:
col = 'Изменение рыночной цены к цене предыдуего дня %)'
dfc[col] = dfc[col].astype(str).replace('-', '0').str.replace(",", ".").astype(float)

In [140]:
col = 'Количество договоров'
dfc[col] = dfc[col].astype(int)

In [141]:
cols = ['Цена в заявках (Лучший спрос)', 
        'Цена в заявках (лучшее предложение)',
        'Цена рын',
        'Цена макс',
        'Цена сред',
        'Цена мин',
        ]
for col in cols:
    dfc[col] = dfc[col].astype(str).replace('-', '0').str.replace(",", ".").astype(float)

In [142]:
cols = ['Объем договоров руб', 
        'Объем договоров в ед'
        ]
for col in cols:
    dfc[col] = dfc[col].astype(str).replace('-', '0').str.replace(",", ".").astype(int)

In [143]:
cols = ['Наименование Инструмента', 
        'Базис поставки'
        ]
for col in cols:
    dfc[col] = dfc[col].str.replace("\n", " ")

In [152]:
dfc['Date'] = pd.to_datetime(latest_file.name.split('.')[0])
dfc.rename(columns={
    'Код инструмента' : 'Code instrument',
    'Наименование Инструмента':'Name instrument',
    'Базис поставки' : 'Basis import',
    'Объем договоров в ед' : 'Volume import(units)',
    'Объем договоров руб' : 'Volume import(rub)',
    'Изменение рыночной цены к цене предыдуего дня руб)' : 'Changes to previous (rub)',
    'Изменение рыночной цены к цене предыдуего дня %)' : 'Changes to previous (%)',
    'Цена мин':'Price min',    
    'Цена сред' : 'Price avg',
    'Цена макс': 'Price max',
    'Цена рын': 'Price market',
    'Цена в заявках (лучшее предложение)': 'Price (best offer)',
    'Цена в заявках (Лучший спрос)': 'Price (Best demand)',
    'Количество договоров' : 'Count contracts',
    'Date' : 'Date',
}, inplace = True) 

C:\Users\Ywur7t\AppData\Local\Temp\ipykernel_15892\810474507.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dfc['Date'] = pd.to_datetime(latest_file.name.split('.')[0])


In [ ]:
dfc.rename(columns={
"Code instrument": "instrument_code",
"Name instrument": "instrument_name",
"Basis import": "delivery_basis",
"Volume import(units)": "volume_import_units",
"Volume import(rub)": "volume_import_rub",
"Changes to previous (rub)": "changes_to_previous_rub",
"Changes to previous (%)": "changes_to_previous_pers",
"Price min": "price_min",
"Price avg": "price_avg",
"Price max": "price_max",
"Price market": "price_market",
"Price (best offer)": "price_best_offer",
"Price (Best demand)": "price_best_demand",
"Count contracts": "count_contracts",
"Date": "date",
}, inplace=True)

In [ ]:
table = pa.Table.from_pandas(dfc)
pq.write_table(table, f"data/staging/{latest_file.name}")

In [168]:
for i in df.columns:
    print(f'- {i}')

- instrument_code
- instrument_name
- delivery_basis
- volume_import_units
- volume_import_rub
- changes_to_previous_rub
- changes_to_previous_pers
- price_min
- price_avg
- price_max
- price_market
- price_best_offer
- price_best_demand
- count_contracts
- date
